# 第 2 章 §2.3 —— Routing 逐行拆解

> 这是书里 `Chapter_02_Routing_(LangGraph)` 的**注释版**。
> 【适配】原版用 `ChatGoogleGenerativeAI` + `GOOGLE_API_KEY`，本机没有 Google
> 凭证，已换成 `common.make_llm()`（DeepSeek，OpenAI 兼容）。
> 逻辑与分支结构保持一致，但 **prompt 已被改动**（见下方清单）。
>
> **文件名有坑**：这个 notebook 叫 `(LangGraph)`，但它**根本没用 LangGraph**，
> 用的是 LangChain 的 `RunnableBranch`。书里真正的 LangGraph 条件边
> （`StateGraph` + `add_conditional_edges`）并没给完整代码。

## 一句话概括

> 先用**一次模型调用**把请求分类成 3 个标签之一，
> 再用 **`RunnableBranch`** 按标签选一条分支去执行。

和第 1 章相比，变的是**控制流**：从「写死的顺序」变成「运行时按内容决定」。

## 数据流全景（最重要的心智模型）

```
invoke({"request": "Book me a flight"})
   │
   ├── {"decision": router_chain,     ← 调 1 次模型，得 "booker"
   │    "request":  Passthrough()}    ← 原样保留输入，一个字节不改
   │        ↓ 合成一个 dict（并行算完再合并）
   │   {"decision": "booker",
   │    "request": {"request": "Book me a flight"}}
   │                    ↑ 这层嵌套就是 x['request']['request'] 的由来
   │
   ├── RunnableBranch 按 decision 选中 branches["booker"]
   │        ↓ 执行 booking_handler
   │   {"decision": ..., "request": ..., "output": "Booking Handler ..."}
   │
   └── lambda x: x['output']          ← 只抛结果字符串出去
```

## 本 notebook 相对书里原版的改动

| # | 改动 | 影响 |
|---|---|---|
| 1 | `info` 分类被**收窄**为「与订机票/酒店相关的信息问题」；原版是「所有其它信息问题」 | 与预订无关的普通知识问题，现在会落到 `unclear` |
| 2 | 演示用例 3 个 → 4 个（B 补了上下文，新增 D） | 覆盖了更多边界 |
| 3 | 修了 prompt 拼写 `destinatuion` → `destination` | 无语义变化 |

> ⚠️ 因为 prompt 已偏离原书，本 notebook 的路由结果**不能**直接和
> `../../ch02_routing/02_3_debug_lab.ipynb`（保留书里原版 prompt）对照。
>
> 📌 实测提醒：收窄后的 prompt 让 D 成为**边界用例**，连跑 6 次
> `booker` 3 次、`info` 3 次（`temperature=0` 也照样摇摆）。
> 详见下方第 ⑦ 步的注释。

想动手调试，用 `../../ch02_routing/02_3_debug_lab.ipynb`（25 格，含 3 个实验）。

In [1]:
# =============================================================================
# 第 2 章 §2.3 完整代码（逐行注释）
# =============================================================================
#
# 和 Ch1 一样，整个脚本分两个阶段：
#   阶段 A（①—⑥）：搭积木，只有 prompt 插值，不发任何网络请求
#   阶段 B（⑦）  ：invoke() 点火，这时才调模型
#
# 唯一的区别是：Ch1 是「一条直线」，Ch2 是「一个岔路口」。

import sys
from pathlib import Path

# 【适配】向上找到含 common/ 的目录再加进 sys.path。
# 不能写死 Path.cwd()：notebook 的 kernel 工作目录可能是仓库根，
# 也可能是 notebook 自己所在的 chapter_notebooks/ —— 取决于 VS Code 的
# jupyter.notebookFileRoot 设置。向上搜索对两种情况都成立。
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
else:
    raise RuntimeError("找不到 common/ 目录，请确认 kernel 工作目录在仓库内")

# 【适配】原版：from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableBranch

from common import make_llm


# ─────────────────────────────────────────────────────────────────────────────
# ① 三个目标 handler —— 路由的下游是「三种完全不同的能力」
# ─────────────────────────────────────────────────────────────────────────────
# 它们现在是「打印 + 返回字符串」，代表真正干活的分支。
# 真实系统里：booking_handler 去调订单库，info_handler 去查知识库，
# unclear_handler 去建人工工单。
#
# 关键认知：路由的价值来自下游的**异质性**。
# 如果三条分支做的事差不多，就根本不需要路由。

def booking_handler(request: str) -> str:
    """模拟 Booking Agent：处理机票/酒店预订。"""
    print("\n--- DELEGATING TO BOOKING HANDLER ---")
    return f"Booking Handler processed request: '{request}'. Result: Simulated booking action."


def info_handler(request: str) -> str:
    """模拟 Info Agent：处理一般信息问答。"""
    print("\n--- DELEGATING TO INFO HANDLER ---")
    return f"Info Handler processed request: '{request}'. Result: Simulated information retrieval."


def unclear_handler(request: str) -> str:
    """兜底：请求无法归类。"""
    print("\n--- HANDLING UNCLEAR REQUEST ---")
    return f"Coordinator could not delegate request: '{request}'. Please clarify."


# ─────────────────────────────────────────────────────────────────────────────
# ② 路由器 prompt —— 整个模式的「大脑」就是这一段字符串
# ─────────────────────────────────────────────────────────────────────────────
# 【本 notebook 相对书里原版的改动】—— 这一处很容易被忽略，但它改变了整个边界
#   书里原版第二行：For all other general information questions, output 'info'.
#   这里被收窄成：只限 *与订机票/酒店相关* 的信息问题（如查目的地、行程规划）。
#
#   后果：info 的边界大幅收缩，凡是「与预订无关的普通知识问题」都会掉进 unclear。
#   实测对比（同一句话，只换 prompt）：
#     "What is the capital of Italy?"
#       原版 prompt  -> 'info'
#       收窄 prompt  -> 'unclear'
#
#   ⚠️ 这就是 prompt 工程最危险的地方：**行为变更没有 diff 可看**。
#   代码一个字没动，只改了一句自然语言，整个分类边界就平移了，
#   而且没有任何测试会因此失败。
#
# 三个必要条件，缺一不可：
#   1. 候选集有限且封闭（只有 3 个值）
#   2. 互斥（一个请求只能落一类）
#   3. 单 token 可表达（只要标签，不要解释）
#
# 注意 from_messages 把 system 和 user 分成**两条消息**。
# 把指令混进 user 消息里，模型更容易被用户输入带偏（prompt injection）。
coordinator_router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze the user's request and determine which specialist handler should process it.
     - If the request is related to booking flights or hotels, output 'booker'.
     - For all other general information questions about booking flights or hotels, for example, checking destination of trip planning, output 'info'.
     - If the request is unclear or doesn't fit either category, output 'unclear'.
     ONLY output one word: 'booker', 'info', or 'unclear'."""),
    ("user", "{request}"),
])


# ─────────────────────────────────────────────────────────────────────────────
# ③ 路由器链：prompt → llm → 字符串
# ─────────────────────────────────────────────────────────────────────────────
# 【适配】原版 llm 用 ChatGoogleGenerativeAI(model="gemini-2.5-flash")。
# 这次调用会花 token，是整条链路里**唯一**的模型调用。
# ⚠️ `ONLY output one word` 是**请求**，不是**约束**。
# 模型可能返回 'Booker'、'booker.'、或一句解释 —— 见第 ⑤ 步的 .strip() 问题。
coordinator_router_chain = coordinator_router_prompt | make_llm(temperature=0) | StrOutputParser()


# ─────────────────────────────────────────────────────────────────────────────
# ④ 分支表：每个决策值 → 一个可执行的 Runnable
# ─────────────────────────────────────────────────────────────────────────────
# RunnablePassthrough.assign(output=...) 的语义：
#   保留上游输入的所有键，再**追加**一个 output 键。
#
# ⚠️ 注意 x['request']['request'] 这层双重嵌套，不是笔误：
#   外层 x['request']  = coordinator 里 RunnablePassthrough() 透传下来的
#                        「原始输入 dict」，即 {"request": "Book me a flight"}
#   内层 ['request']   = 真正的用户文本 "Book me a flight"
# 这是 LCEL「并行打包」造成的形状。读代码时最容易卡在这里。
branches = {
    "booker": RunnablePassthrough.assign(
        output=lambda x: booking_handler(x["request"]["request"])
    ),
    "info": RunnablePassthrough.assign(
        output=lambda x: info_handler(x["request"]["request"])
    ),
    "unclear": RunnablePassthrough.assign(
        output=lambda x: unclear_handler(x["request"]["request"])
    ),
}


# ─────────────────────────────────────────────────────────────────────────────
# ⑤ RunnableBranch：条件选路
# ─────────────────────────────────────────────────────────────────────────────
# 语法是 (谓词, 分支) 成对，**最后一对只写分支** —— 它就是默认分支。
#
# ⚠️ 书里用 .strip()，只能去首尾空白。这些语义正确的输出都会失败：
#     'booker.'      → 落到默认分支
#     'Booker'       → 大写不符，落到默认分支
#     "'booker'"     → 带引号，落到默认分支
#   而默认分支是 unclear_handler，于是系统礼貌地回答「请你说清楚一点」。
#   **不报错、日志正常、用户被无意义地反复追问。**
#   更鲁棒的归一化写法见 02_3_debug_lab.ipynb 第 5 节。
delegation_branch = RunnableBranch(
    (lambda x: x["decision"].strip() == "booker", branches["booker"]),
    (lambda x: x["decision"].strip() == "info", branches["info"]),
    branches["unclear"],          # 默认分支：'unclear' 以及**任何意外输出**
)


# ─────────────────────────────────────────────────────────────────────────────
# ⑥ 组装 coordinator：先算决策，再分派，最后只取结果
# ─────────────────────────────────────────────────────────────────────────────
# {"decision": ..., "request": ...} 这个 dict 和 Ch1 的
# {"specifications": ...} 是**同一个机制**（RunnableParallel）：
# 两个值并行算出来，再合成一个 dict。
#
#   decision → 调 1 次模型做分类
#   request  → RunnablePassthrough()，原样保留
#
# 为什么必须留一份原件？因为「分类」和「最终处理」都要用同一段用户文本。
# 分类器用完就丢的话，下游 handler 就没输入了。
coordinator_agent = (
    {
        "decision": coordinator_router_chain,
        "request": RunnablePassthrough(),
    }
    | delegation_branch
    | (lambda x: x["output"])     # 只把结果抛出去，丢掉中间过程
)


# ─────────────────────────────────────────────────────────────────────────────
# ⑦ 点火：跑三个演示用例
# ─────────────────────────────────────────────────────────────────────────────
# 注意这四个用例**没有任何断言**，只 print。
# 所以「跑完没报错」≠「路由对了」——
# 用例名字里的 booker/info/unclear 是**人写的期望标签**，不是模型的决定。
#
# 实测结果（deepseek-flash, temperature=0, 本 notebook 收窄后的 prompt）：
#   A "Book me a flight to London."                        → booker       稳定 ✅ 与标签一致
#   B "What is the capital of Italy? I`m planning a trip."   → info         6/6 稳定 ✅ 一致
#   C "Tell me about quantum physics."                     → unclear      稳定 ✅ 与标签一致
#   D "Tell me the earliest flight I can catch from ..."    → booker/info  各占一半 ⚠️ 标签写 info
#
# 三个值得停下来看的地方：
#
# (1) 用例 B 是**靠补上下文才**走进 info 的。
#     同一句话去掉 "I`m planning a trip." 之后，实测会落到 unclear ——
#     因为收窄后的 prompt 要求 info 必须与订机票/酒店相关。
#     **一句话的上下文，决定了它走哪条分支。**
#
# (2) 用例 D **不稳定**：连跑 6 次，booker 3 次、info 3 次（temperature=0！）。
#     原因是这句话本身跨在两类边界上：
#       "查询最早能赶上的航班" —— 算「预订动作」，还是算「信息查询」？
#     prompt 没定义清楚，而 deepseek-flash 的推理路径每次也不完全相同，
#     于是结果来回摇摆。
#     ⚠️ 这类不稳定最危险：**每次运行都"成功"，日志里找不到任何异常**。
#     你只会看到用户时而被订票、时而被查信息。
#
# (3) 「标签」和「模型决定」是两回事。
#     用例名里的 booker/info/unclear 是**人写的期望**；模型给的是它自己的判断。
#     这段演示**没有任何断言**去比较两者 —— 这正是 Ch19（评测）要解决的问题。
#
# 书里原版只有 3 个用例，且其中的"C"实际走的是 info（见 lab notebook 第 8 节）。
# 本 notebook 收窄 prompt 后，C 才真正落到 unclear。
for name, request in [
    ("A booking", "Book me a flight to London."),
    ("B info", "What is the capital of Italy? I`m planning a trip."),
    ("C unclear (book label)", "Tell me about quantum physics."),
    ("D info", "Tell me the earliest flight I can catch from New York to Roma next week."),
]:
    print(f"\n=== Running: {name} ===")
    print(f"Final Result: {coordinator_agent.invoke({'request': request})}")


=== Running: A booking ===



--- DELEGATING TO BOOKING HANDLER ---
Final Result: Booking Handler processed request: 'Book me a flight to London.'. Result: Simulated booking action.

=== Running: B info ===



--- DELEGATING TO INFO HANDLER ---
Final Result: Info Handler processed request: 'What is the capital of Italy? I`m planning a trip.'. Result: Simulated information retrieval.

=== Running: C unclear (book label) ===



--- HANDLING UNCLEAR REQUEST ---
Final Result: Coordinator could not delegate request: 'Tell me about quantum physics.'. Please clarify.

=== Running: D info ===



--- DELEGATING TO INFO HANDLER ---
Final Result: Info Handler processed request: 'Tell me the earliest flight I can catch from New York to Roma next week.'. Result: Simulated information retrieval.


## 执行时序：一次 `invoke()` 内部到底发生了什么

以 `"Book me a flight to London."` 为例：

| # | 环节 | 类型 | 花 token | 数据 |
|---|---|---|---|---|
| 1 | dict 展开 `{"decision":…, "request":…}` | 本地（构造并行） | ❌ | `{"request": "Book me…"}` |
| 2 | `Passthrough()` 分支 | 本地 | ❌ | 原样 `{"request": "Book me…"}` |
| 3 | 路由 prompt 插值 | 本地 | ❌ | → 2 条消息（system + user） |
| 4 | **`llm` 分类调用** | **网络** | ✅ | → `AIMessage("booker")` |
| 5 | `StrOutputParser` | 本地 | ❌ | → `"booker"` |
| 6 | dict 合并 | 本地 | ❌ | `{"decision": "booker", "request": {"request": "Book me…"}}` |
| 7 | `RunnableBranch` 谓词求值 | 本地 | ❌ | 选中 `branches["booker"]` |
| 8 | `branches["booker"]` | 本地 | ❌ | 调 handler → 追加 `output` 键 |
| 9 | `lambda x: x["output"]` | 本地 | ❌ | → 结果字符串 |

**关键数字**：整条路由链路**只花 1 次模型调用**。
这比 Ch1 的 2 次还省——因为路由只需要一个标签，不需要生成内容。

**关键风险**：第 4 步是唯一的模型调用，它错了，后面 5 步全都"正确地"执行错误决定。

## 八个关键点（书里没明说，但决定你怎么用它）

### ① 路由是「分类问题」，不是「生成问题」

模型在这里只输出一个标签，不生成内容。这意味着：

- 可以用**小模型**甚至规则来做，成本极低
- 可以单独评测（准确率、混淆矩阵），比评测生成质量容易得多
- 可以用**非 LLM** 的方案替代（书里列的 Embedding / 规则 / ML 分类器）

### ② 四种路由机制是**成本阶梯**，不是四选一

书里列了 LLM / Embedding / 规则 / ML 分类器。真实系统几乎都是**混合**的：

```
最常见格式  → 规则（~0 成本，覆盖 60-70%）
语义相近    → Embedding（本地向量比较）
长尾/模糊   → LLM 分类（1 次 API 调用）
```

先用免费的挡掉大多数，贵的那层只兜底。

### ③ `RunnablePassthrough()` 是「留一份原件」

分类和处理都要用同一段用户文本，所以必须原样透传。
这个模式在 agent 里极常见（Ch7 多 agent、Ch8 记忆都会再见到）。

### ④ 双重嵌套 `x['request']['request']` 的由来

不是笔误，是 `{"request": RunnablePassthrough()}` 把「原始输入 dict」
整个透传下来造成的。理解了这一点，LCEL 的数据形状就通了。

### ⑤ `.strip()` 是脆弱的防御

只能去空白。`'booker.'`、`'Booker'`、`"'booker'"` 全部会落到默认分支。
**正确做法是从严格到宽松的多级归一化**（精确 → 去包裹 → 文本内唯一命中 → 兜底）。

### ⑥ 默认分支承载了两种不同语义

```python
branches["unclear"]   # 默认分支
```

它同时接收：

- 「模型**明确判定**为 unclear」→ 正常业务路径
- 「模型输出了**意料之外**的任何东西」→ **异常信号，应该告警**

两者合并成一个，就等于**把异常伪装成正常业务**。生产环境必须拆开。

### ⑦ 路由错误是「静默降级」，比 Ch1 更隐蔽

Ch1 的错误至少可能表现为"格式不对"。路由错了，系统会**大方地返回一个
看起来很正常的回复**，只是答非所问。日志里一切正常。

> 这是 agent 可靠性最难的部分：**错误的执行路径不会自己喊疼。**

### ⑧ 类别边界上的样本会**摇摆**（实测）

`"Tell me the earliest flight I can catch from New York to Roma next week."`
连跑 6 次：`booker` 3 次、`info` 3 次。**`temperature=0` 也挡不住。**

原因是两类定义在语义上重叠了：这句话既是「预订相关的动作」，也是「信息查询」。
prompt 没给出优先级，模型的推理路径又有细微随机性，于是五五开。

这类问题的诊断难度远高于崩溃：

- 不会报错，不会降级，每次输出都合理
- 复现困难（再跑一次可能就"好了"）
- 只有**多次采样 + 统计**才能发现

> 工程对策：对边界样本做**主动采样**（同一输入跑 N 次看一致性），
> 发现摇摆就把该类别拆细或补规则 —— 这正是 Ch19 评测要做的事。

## 书里没有的两个结构性缺陷

### 缺陷 A：单标签放不下复合意图

`"Book me a flight AND tell me about Italy"` 既是预订也是信息。
但输出空间只有一个标签。实测返回 `'booker'`，**信息那半被静默丢弃**。

这是**路由粒度设计问题**，不是模型能力问题。解法有三种：

1. 识别复合意图 → 拆成子任务（Ch6 Planning）
2. 多标签路由 → 输出列表 → 并行执行
3. 加一个 `multi` 类别 → 交给协调者

### 缺陷 B：没有回退路径

一旦分派出去，handler 无法说"我处理不了，退回协调者"。
真实系统需要 ReAct 式的**回退边**（Ch6、Ch12）。

## 想生产化还缺什么

| 缺什么 | 后果 | 正解在哪章 |
|---|---|---|
| 输出归一化 | 格式稍变即静默降级 | 本节第 ⑤ 点 / Ch18 |
| 意外输出的告警 | 异常伪装成正常业务 | Ch18 Guardrails |
| 复合请求分解 | 一半意图丢失 | Ch6 Planning |
| 路由正确性评测 | 不知道准确率多少 | Ch19 Evaluation |
| 回退机制 | 分类错了全错 | Ch12 Exception Handling |